<a href="https://colab.research.google.com/github/abubakarshahid439/ABUBAKAR.flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abubakarshahid439/ABUBAKAR.flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

One row represents one content item observation from the provided starter dataset. The dataset is a single snapshot rather than a longitudinal dataset covering multiple dates. Therefore, the analysis focuses on relationships between search, content, ranking, and performance signals within this snapshot.

In [ ]:
import pandas as pd

import pandas as pd

url = "https://raw.githubusercontent.com/abubakarshahid439/ABUBAKAR.flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Total rows:", len(df))
print("Total columns:", len(df.columns))


Features
search_volume
competition
cpc
word_count
char_count
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
trend_pct
Label / outcome

There is no direct binary clicked label in the provided dataset. Therefore, I will not create an artificial clicked label. For this stage, observed performance signals such as ctr can be treated as an outcome signal for investigation.

Context
content_type
main_intent
competition_level
trend_direction
impression_tier
position_tier
Excluded

Identifier fields such as content_id and client_id are excluded from predictive features because they identify records rather than describe ranking or performance behavior.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Check shape (rows, columns)
print("Shape:", df.shape)

# Check missing values
print("\nMissing values:\n", df.isnull().sum())

# Check shape
print("Shape:", df.shape)

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Check unique content items
print("\nUnique content items:", df["content_id"].nunique())

# Check unique clients
print("Unique clients:", df["client_id"].nunique())

# Check important numeric fields
print("\nCTR mean:", df["ctr"].mean())
print("CTR median:", df["ctr"].median())
print("Median average position:", df["avg_position"].median())

# Check available columns
print("\nColumns:")
print(df.columns.tolist())

This dataset is a single snapshot and does not provide a complete history of ranking changes over time. It also does not contain a direct user-level click label, so I cannot directly model individual click decisions from this data. The observed relationships between ranking position, CTR, engagement, and other signals do not prove causation. The dataset may also not generalize to all search systems, clients, queries, or users. Therefore, the results should be treated as directional evidence and decision support for ranking investigation rather than proof of how a search engine ranks results.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check whether a direct click label exists
print("Is 'clicked' available?", "clicked" in df.columns)

# Check whether date information exists
print("Is 'date' available?", "date" in df.columns)

# Check number of rows available for analysis
print("Rows available for analysis:", len(df))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

In [ ]:
!pip -q install duckdb

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB + Hugging Face connection ready")

DuckDB + Hugging Face connection ready


In [ ]:
files = con.execute("""
    SELECT *
    FROM glob(
        'hf://datasets/FlyRank/internship-warehouse/**/*.parquet'
    )
""").fetchall()

for f in files:
    print(f[0])

In [ ]:
march_file = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{march_file}')"
).fetchdf()

display(schema)

In [ ]:
grain_check = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{march_file}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    LIMIT 10
""").fetchdf()

display(grain_check)

In [ ]:
query2 = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{march_file}')
""").fetchdf()

display(query2)

In [ ]:
query3 = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{march_file}')
""").fetchdf()

display(query3)

In [ ]:
full_schema = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{march_file}')
""").fetchdf()

display(full_schema)

In [ ]:
sample = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        scroll_events
    FROM read_parquet('{march_file}')
    LIMIT 10
""").fetchdf()

display(sample)

In [ ]:
availability_check = con.execute(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS rows,
        COUNT(gsc_impressions) AS impressions_present,
        COUNT(gsc_clicks) AS clicks_present,
        ga4_data_available,
        COUNT(ga4_sessions) AS sessions_present,
        COUNT(scroll_events) AS scroll_present
    FROM read_parquet('{march_file}')
    GROUP BY
        gsc_data_available,
        ga4_data_available
    ORDER BY
        gsc_data_available,
        ga4_data_available
""").fetchdf()

display(availability_check)

In [ ]:
feature_check = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,

        LAG(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS prev_gsc_impressions,

        LAG(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS prev_gsc_clicks,

        LAG(gsc_avg_position) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS prev_gsc_avg_position,

        LAG(ga4_sessions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS prev_ga4_sessions,

        LAG(scroll_events) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS prev_scroll_events

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    LIMIT 20
""").fetchdf()

display(feature_check)

In [ ]:
months_check = con.execute("""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS distinct_days,
        COUNT(*) AS total_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
""").fetchdf()

display(months_check)

In [ ]:
ctr_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_impressions > 0
        ) AS rows_with_impressions,
        COUNT(*) FILTER (
            WHERE gsc_impressions > 0
            AND gsc_clicks IS NOT NULL
        ) AS rows_with_ctr_possible,
        MIN(gsc_impressions) AS min_impressions,
        MAX(gsc_impressions) AS max_impressions,
        MIN(gsc_clicks) AS min_clicks,
        MAX(gsc_clicks) AS max_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
""").fetchdf()

display(ctr_check)

In [ ]:
ctr_validation = con.execute("""
    SELECT
        COUNT(*) AS valid_rows,

        COUNT(*) FILTER (
            WHERE gsc_clicks > gsc_impressions
        ) AS clicks_greater_than_impressions,

        MIN(
            CAST(gsc_clicks AS DOUBLE) / NULLIF(gsc_impressions, 0)
        ) AS min_ctr,

        MAX(
            CAST(gsc_clicks AS DOUBLE) / NULLIF(gsc_impressions, 0)
        ) AS max_ctr,

        AVG(
            CAST(gsc_clicks AS DOUBLE) / NULLIF(gsc_impressions, 0)
        ) AS mean_ctr,

        MEDIAN(
            CAST(gsc_clicks AS DOUBLE) / NULLIF(gsc_impressions, 0)
        ) AS median_ctr

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_impressions > 0
""").fetchdf()

display(ctr_validation)

In [ ]:
model_data = con.execute("""
    WITH daily AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,

            gsc_clicks,
            gsc_impressions,

            gsc_avg_position,
            ga4_sessions,
            ga4_engaged_sessions,
            ga4_total_engagement_sec,
            scroll_events

        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
        )
    ),

    lagged AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,

            gsc_clicks,
            gsc_impressions,

            LAG(gsc_avg_position) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_gsc_avg_position,

            LAG(ga4_sessions) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_ga4_sessions,

            LAG(ga4_engaged_sessions) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_ga4_engaged_sessions,

            LAG(ga4_total_engagement_sec) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_ga4_total_engagement_sec,

            LAG(scroll_events) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_scroll_events

        FROM daily
    )

    SELECT
        report_date,
        client_hash_id,
        content_hash_id,

        CAST(gsc_clicks AS DOUBLE)
            / NULLIF(gsc_impressions, 0) AS ctr,

        prev_gsc_avg_position,
        prev_ga4_sessions,
        prev_ga4_engaged_sessions,
        prev_ga4_total_engagement_sec,
        prev_scroll_events

    FROM lagged

    WHERE gsc_impressions > 0
""").fetchdf()

print("Rows:", len(model_data))
display(model_data.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
model_data = con.execute("""
    WITH daily AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_clicks,
            gsc_impressions,
            gsc_avg_position,
            ga4_sessions,
            ga4_engaged_sessions,
            ga4_total_engagement_sec,
            scroll_events
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
        )
        WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
    ),

    lagged AS (
        SELECT
            *,
            LAG(gsc_avg_position) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_gsc_avg_position,

            LAG(ga4_sessions) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_ga4_sessions,

            LAG(ga4_engaged_sessions) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_ga4_engaged_sessions,

            LAG(ga4_total_engagement_sec) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_ga4_total_engagement_sec,

            LAG(scroll_events) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_scroll_events

        FROM daily
    )

    SELECT
        report_date,
        client_hash_id,
        content_hash_id,

        CAST(gsc_clicks AS DOUBLE)
            / NULLIF(gsc_impressions, 0) AS ctr,

        prev_gsc_avg_position,
        prev_ga4_sessions,
        prev_ga4_engaged_sessions,
        prev_ga4_total_engagement_sec,
        prev_scroll_events

    FROM lagged
    WHERE gsc_impressions > 0
    LIMIT 500000
""").fetchdf()

print("Rows:", len(model_data))
display(model_data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 500000


,report_date,client_hash_id,content_hash_id,ctr,prev_gsc_avg_position,prev_ga4_sessions,prev_ga4_engaged_sessions,prev_ga4_total_engagement_sec,prev_scroll_events
0,2026-01-01,client_0797ff3a1fc9a6a5,content_b547142084c65b0e,0.0,NaN,<NA>,<NA>,<NA>,<NA>
1,2026-01-02,client_0797ff3a1fc9a6a5,content_b547142084c65b0e,0.0,11.142857,<NA>,<NA>,<NA>,<NA>
2,2026-01-03,client_0797ff3a1fc9a6a5,content_b547142084c65b0e,0.0,10.000000,<NA>,<NA>,<NA>,<NA>
3,2026-01-04,client_0797ff3a1fc9a6a5,content_b547142084c65b0e,0.0,10.250000,<NA>,<NA>,<NA>,<NA>
4,2026-01-05,client_0797ff3a1fc9a6a5,content_b547142084c65b0e,0.0,10.000000,<NA>,<NA>,<NA>,<NA>


In [4]:
model_data = con.execute("""
    WITH daily AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_clicks,
            gsc_impressions,
            gsc_avg_position,
            ga4_sessions,
            ga4_engaged_sessions,
            ga4_total_engagement_sec,
            scroll_events
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
        )
        WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
    ),

    lagged AS (
        SELECT
            *,
            LAG(gsc_avg_position) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_gsc_avg_position,

            LAG(ga4_sessions) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_ga4_sessions,

            LAG(ga4_engaged_sessions) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_ga4_engaged_sessions,

            LAG(ga4_total_engagement_sec) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_ga4_total_engagement_sec,

            LAG(scroll_events) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS prev_scroll_events

        FROM daily
    )

    SELECT
        report_date,
        client_hash_id,
        content_hash_id,

        CAST(gsc_clicks AS DOUBLE)
            / NULLIF(gsc_impressions, 0) AS ctr,

        prev_gsc_avg_position,
        prev_ga4_sessions,
        prev_ga4_engaged_sessions,
        prev_ga4_total_engagement_sec,
        prev_scroll_events

    FROM lagged
    WHERE gsc_impressions > 0
    LIMIT 500000
""").fetchdf()

print("Rows:", len(model_data))
display(model_data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 500000


,report_date,client_hash_id,content_hash_id,ctr,prev_gsc_avg_position,prev_ga4_sessions,prev_ga4_engaged_sessions,prev_ga4_total_engagement_sec,prev_scroll_events
0,2026-01-02,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0.0,NaN,<NA>,<NA>,<NA>,<NA>
1,2026-01-09,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0.0,NaN,<NA>,<NA>,<NA>,<NA>
2,2026-01-29,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0.0,NaN,<NA>,<NA>,<NA>,<NA>
3,2026-01-31,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0.0,NaN,<NA>,<NA>,<NA>,<NA>
4,2026-01-01,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,0.0,NaN,<NA>,<NA>,<NA>,<NA>


In [6]:
feature_availability = model_data[
    [
        "prev_gsc_avg_position",
        "prev_ga4_sessions",
        "prev_ga4_engaged_sessions",
        "prev_ga4_total_engagement_sec",
        "prev_scroll_events"
    ]
].notna().sum().to_frame("non_null_rows")

feature_availability["missing_rows"] = (
    len(model_data) - feature_availability["non_null_rows"]
)

feature_availability["availability_pct"] = (
    feature_availability["non_null_rows"]
    / len(model_data) * 100
).round(2)

display(feature_availability)

,non_null_rows,missing_rows,availability_pct
prev_gsc_avg_position,443356,56644,88.67
prev_ga4_sessions,293460,206540,58.69
prev_ga4_engaged_sessions,293460,206540,58.69
prev_ga4_total_engagement_sec,293460,206540,58.69
prev_scroll_events,293460,206540,58.69


In [7]:
missingness_check = con.execute("""
    SELECT
        CASE
            WHEN ga4_sessions IS NULL
            THEN 'GA4 missing'
            ELSE 'GA4 available'
        END AS ga4_status,

        COUNT(*) AS rows,

        AVG(
            CAST(gsc_clicks AS DOUBLE)
            / NULLIF(gsc_impressions, 0)
        ) AS mean_ctr,

        MEDIAN(
            CAST(gsc_clicks AS DOUBLE)
            / NULLIF(gsc_impressions, 0)
        ) AS median_ctr

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )

    WHERE
        gsc_impressions > 0
        AND report_date BETWEEN '2026-01-01' AND '2026-03-31'

    GROUP BY
        ga4_status

    ORDER BY
        ga4_status
""").fetchdf()

display(missingness_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_status,rows,mean_ctr,median_ctr
0,GA4 available,4455211,0.003271,0.0
1,GA4 missing,4174776,0.003095,0.0


In [8]:
import pandas as pd

features = [
    "prev_gsc_avg_position",
    "prev_ga4_sessions",
    "prev_ga4_engaged_sessions",
    "prev_ga4_total_engagement_sec",
    "prev_scroll_events"
]

# Keep only rows with a valid target
clean_data = model_data.dropna(subset=["ctr"]).copy()

# Median imputation for the 5 features
for col in features:
    clean_data[col] = clean_data[col].fillna(
        clean_data[col].median()
    )

print("Original rows:", len(model_data))
print("Clean rows:", len(clean_data))

print("\nRemaining missing values:")
display(clean_data[features].isna().sum())

display(clean_data.head())

Original rows: 500000
Clean rows: 500000

Remaining missing values:


,0
prev_gsc_avg_position,0
prev_ga4_sessions,0
prev_ga4_engaged_sessions,0
prev_ga4_total_engagement_sec,0
prev_scroll_events,0


,report_date,client_hash_id,content_hash_id,ctr,prev_gsc_avg_position,prev_ga4_sessions,prev_ga4_engaged_sessions,prev_ga4_total_engagement_sec,prev_scroll_events
0,2026-01-02,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0.0,7.097015,0,0,0,0
1,2026-01-09,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0.0,7.097015,0,0,0,0
2,2026-01-29,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0.0,7.097015,0,0,0,0
3,2026-01-31,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0.0,7.097015,0,0,0,0
4,2026-01-01,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,0.0,7.097015,0,0,0,0


In [9]:
model_data = model_data.sort_values("report_date").reset_index(drop=True)

n = len(model_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = model_data.iloc[:train_end].copy()
val = model_data.iloc[train_end:val_end].copy()
test = model_data.iloc[val_end:].copy()

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

print("\nDate ranges:")
print("Train:", train["report_date"].min(), "→", train["report_date"].max())
print("Validation:", val["report_date"].min(), "→", val["report_date"].max())
print("Test:", test["report_date"].min(), "→", test["report_date"].max())

Train: 350000
Validation: 75000
Test: 75000

Date ranges:
Train: 2026-01-01 00:00:00 → 2026-03-12 00:00:00
Validation: 2026-03-12 00:00:00 → 2026-03-22 00:00:00
Test: 2026-03-22 00:00:00 → 2026-03-31 00:00:00


In [10]:
model_data = model_data.sort_values(
    ["report_date", "client_hash_id", "content_hash_id"]
).reset_index(drop=True)

train = model_data[
    model_data["report_date"] < "2026-03-15"
].copy()

val = model_data[
    (model_data["report_date"] >= "2026-03-15") &
    (model_data["report_date"] < "2026-03-23")
].copy()

test = model_data[
    model_data["report_date"] >= "2026-03-23"
].copy()

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

print("\nDate ranges:")
print("Train:", train["report_date"].min(), "→", train["report_date"].max())
print("Validation:", val["report_date"].min(), "→", val["report_date"].max())
print("Test:", test["report_date"].min(), "→", test["report_date"].max())

Train: 368236
Validation: 60851
Test: 70913

Date ranges:
Train: 2026-01-01 00:00:00 → 2026-03-14 00:00:00
Validation: 2026-03-15 00:00:00 → 2026-03-22 00:00:00
Test: 2026-03-23 00:00:00 → 2026-03-31 00:00:00


In [11]:
features = [
    "prev_gsc_avg_position",
    "prev_ga4_sessions",
    "prev_ga4_engaged_sessions",
    "prev_ga4_total_engagement_sec",
    "prev_scroll_events"
]

# Calculate medians ONLY from training data
train_medians = {}

for col in features:
    train_medians[col] = train[col].median()

print("Training medians:")
for col, value in train_medians.items():
    print(f"{col}: {value}")

# Apply training medians to all splits
for col in features:
    train[col] = train[col].fillna(train_medians[col])
    val[col] = val[col].fillna(train_medians[col])
    test[col] = test[col].fillna(train_medians[col])

print("\nMissing values after imputation:")

print("\nTrain:")
display(train[features].isna().sum())

print("\nValidation:")
display(val[features].isna().sum())

print("\nTest:")
display(test[features].isna().sum())

Training medians:
prev_gsc_avg_position: 6.863636363636363
prev_ga4_sessions: 0.0
prev_ga4_engaged_sessions: 0.0
prev_ga4_total_engagement_sec: 0.0
prev_scroll_events: 0.0

Missing values after imputation:

Train:


,0
prev_gsc_avg_position,0
prev_ga4_sessions,0
prev_ga4_engaged_sessions,0
prev_ga4_total_engagement_sec,0
prev_scroll_events,0



Validation:


,0
prev_gsc_avg_position,0
prev_ga4_sessions,0
prev_ga4_engaged_sessions,0
prev_ga4_total_engagement_sec,0
prev_scroll_events,0



Test:


,0
prev_gsc_avg_position,0
prev_ga4_sessions,0
prev_ga4_engaged_sessions,0
prev_ga4_total_engagement_sec,0
prev_scroll_events,0


In [13]:
from sklearn.metrics import mean_absolute_error

# Training-set mean CTR
baseline_ctr = train["ctr"].mean()

# Predict the same value for every row
val_baseline_pred = [baseline_ctr] * len(val)
test_baseline_pred = [baseline_ctr] * len(test)

# MAE
val_baseline_mae = mean_absolute_error(
    val["ctr"],
    val_baseline_pred
)

test_baseline_mae = mean_absolute_error(
    test["ctr"],
    test_baseline_pred
)

print(f"Training mean CTR: {baseline_ctr:.6f}")
print(f"Baseline Validation MAE: {val_baseline_mae:.6f}")
print(f"Baseline Test MAE: {test_baseline_mae:.6f}")

Training mean CTR: 0.004003
Baseline Validation MAE: 0.006734
Baseline Test MAE: 0.006128


In [14]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# Features
X_train = train[features]
y_train = train["ctr"]

X_val = val[features]
y_val = val["ctr"]

X_test = test[features]
y_test = test["ctr"]

# Model
model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Predictions
val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

# MAE
val_mae = mean_absolute_error(y_val, val_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print(f"Baseline Validation MAE: {val_baseline_mae:.6f}")
print(f"ML Validation MAE:       {val_mae:.6f}")

print()

print(f"Baseline Test MAE:       {test_baseline_mae:.6f}")
print(f"ML Test MAE:             {test_mae:.6f}")

Baseline Validation MAE: 0.006734
ML Validation MAE:       0.006643

Baseline Test MAE:       0.006128
ML Test MAE:             0.005947


In [15]:
import pandas as pd
import numpy as np

results = pd.DataFrame({
    "actual_ctr": y_test.values,
    "predicted_ctr": test_pred
})

results["absolute_error"] = (
    results["actual_ctr"] - results["predicted_ctr"]
).abs()

print("Prediction summary:")
display(results.describe())

print("\nLargest errors:")
display(
    results.sort_values(
        "absolute_error",
        ascending=False
    ).head(10)
)

Prediction summary:


,actual_ctr,predicted_ctr,absolute_error
count,70913.000000,70913.000000,7.091300e+04
mean,0.002918,0.003874,5.946551e-03
std,0.030115,0.001775,2.950553e-02
min,0.000000,0.001763,8.779727e-07
25%,0.000000,0.002903,2.795667e-03
50%,0.000000,0.003589,3.555014e-03
75%,0.000000,0.003962,4.302158e-03
max,1.000000,0.061336,9.982370e-01



Largest errors:


,actual_ctr,predicted_ctr,absolute_error
15688,1.0,0.001763,0.998237
543,1.0,0.002071,0.997929
19281,1.0,0.002231,0.997769
6407,1.0,0.002796,0.997204
40094,1.0,0.002860,0.997140
29001,1.0,0.003367,0.996633
9279,1.0,0.003774,0.996226
25616,1.0,0.003849,0.996151
25140,1.0,0.003849,0.996151
44223,1.0,0.003849,0.996151


In [16]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    model,
    X_val,
    y_val,
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": importance.importances_mean,
    "std": importance.importances_std
}).sort_values(
    "importance",
    ascending=False
)

display(feature_importance)

,feature,importance,std
1,prev_ga4_sessions,1.946193e-04,3.174366e-06
0,prev_gsc_avg_position,9.530879e-05,2.827918e-06
3,prev_ga4_total_engagement_sec,1.448759e-05,4.055127e-07
4,prev_scroll_events,1.019509e-05,3.454211e-07
2,prev_ga4_engaged_sessions,-2.738621e-07,1.036703e-08


Feature importance analysis shows that previous-period GA4 sessions were the most informative feature for predicting CTR, followed by previous GSC average position. Previous GA4 total engagement time and scroll events provided smaller contributions. Previous GA4 engaged sessions had near-zero permutation importance, suggesting that it added little incremental predictive value beyond the other features.

In [17]:
error_analysis = pd.DataFrame({
    "actual": y_test.values,
    "predicted": test_pred
})

error_analysis["abs_error"] = (
    error_analysis["actual"] -
    error_analysis["predicted"]
).abs()

zero_ctr = error_analysis["actual"] == 0
nonzero_ctr = error_analysis["actual"] > 0

print("Zero CTR rows:", zero_ctr.sum())
print("Non-zero CTR rows:", nonzero_ctr.sum())

print("\nMAE for zero CTR:")
print(
    mean_absolute_error(
        error_analysis.loc[zero_ctr, "actual"],
        error_analysis.loc[zero_ctr, "predicted"]
    )
)

print("\nMAE for non-zero CTR:")
print(
    mean_absolute_error(
        error_analysis.loc[nonzero_ctr, "actual"],
        error_analysis.loc[nonzero_ctr, "predicted"]
    )
)

Zero CTR rows: 63237
Non-zero CTR rows: 7676

MAE for zero CTR:
0.003811455091106783

MAE for non-zero CTR:
0.02353605926957271


The model performs substantially better on zero-CTR observations than on non-zero CTR observations. The higher error for non-zero CTR cases is consistent with the highly skewed and zero-inflated nature of the target variable. A small number of high-CTR observations, including CTR values close to 1, contribute disproportionately to prediction error.

**# Random Forest comparison**    Random Forest's impurity-based feature importance identified previous GSC average position as the dominant feature, accounting for approximately 78% of the model's total feature importance. Previous GA4 sessions contributed approximately 14%, while engagement time, scroll events, and engaged sessions had smaller contributions.

In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")

rf_model.fit(X_train, y_train)

rf_val_pred = rf_model.predict(X_val)
rf_test_pred = rf_model.predict(X_test)

rf_val_mae = mean_absolute_error(y_val, rf_val_pred)
rf_test_mae = mean_absolute_error(y_test, rf_test_pred)

print("\nResults:")
print(f"Baseline Validation MAE:       {val_baseline_mae:.6f}")
print(f"HistGradientBoosting Val MAE:  {val_mae:.6f}")
print(f"Random Forest Validation MAE:  {rf_val_mae:.6f}")

print()

print(f"Baseline Test MAE:             {test_baseline_mae:.6f}")
print(f"HistGradientBoosting Test MAE: {test_mae:.6f}")
print(f"Random Forest Test MAE:        {rf_test_mae:.6f}")

Training Random Forest...

Results:
Baseline Validation MAE:       0.006734
HistGradientBoosting Val MAE:  0.006643
Random Forest Validation MAE:  0.006603

Baseline Test MAE:             0.006128
HistGradientBoosting Test MAE: 0.005947
Random Forest Test MAE:        0.005912


In [19]:
rf_importance = pd.DataFrame({
    "feature": features,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(rf_importance)

,feature,importance
0,prev_gsc_avg_position,0.780001
1,prev_ga4_sessions,0.141526
3,prev_ga4_total_engagement_sec,0.056779
4,prev_scroll_events,0.011902
2,prev_ga4_engaged_sessions,0.009792


CTR distribution

In [20]:
ctr_distribution = pd.DataFrame({
    "metric": [
        "Total test rows",
        "Zero CTR rows",
        "Non-zero CTR rows",
        "CTR = 1 rows"
    ],
    "count": [
        len(y_test),
        (y_test == 0).sum(),
        (y_test > 0).sum(),
        (y_test == 1).sum()
    ]
})

ctr_distribution["percentage"] = (
    ctr_distribution["count"] / len(y_test) * 100
).round(2)

display(ctr_distribution)

,metric,count,percentage
0,Total test rows,70913,100.00
1,Zero CTR rows,63237,89.18
2,Non-zero CTR rows,7676,10.82
3,CTR = 1 rows,44,0.06


A 500,000-row modeling sample was used for the initial ML experiment after validating the warehouse structure and data quality.

**Completed**                          ✅ Warehouse connection
✅ Files/tables discovery
✅ March 2026 schema verification
✅ Grain verification
✅ Row count + date span
✅ GSC/GA4 availability checks
✅ CTR validation
✅ Leakage-safe lag features
✅ 5 required features
✅ Missing-value analysis + train-only imputation
✅ Chronological Train/Validation/Test split
✅ Baseline model
✅ HistGradientBoosting
✅ Random Forest
✅ Model comparison
✅ Feature importance
✅ Error analysis
✅ CTR distribution analysis
✅ Final best model identified